# T3.3 EIT-003 Lifecycle Environmental Performance Model

READ.ME 

This notebook is the **shared foundation** for the EIT-003 and EIT-002 web services. It contains everything that is independent of runtime event data and must be run once before EIT-001 or EIT-002 can be used.

- Environment setup (Brightway project initialisation ecoinvent and BONSAI database installation)
- Foreground model: REED foreground database imported from `REED_model_*.xlsx`

### 1. Install background 

In [1]:
import bw2data as bd
import bw2calc as bc
import bw2io as bi
from pathlib import Path
import pandas as pd
import numpy as np
import seaborn as sns
import bw2analyzer as bwa
from reed.utils import CONFIG
import bw_processing as bwp
import sys
import os
from bw2io.strategies.generic import set_code_by_activity_hash

sys.path.insert(0, os.path.abspath('../src/reed'))

09:26:00+0200 [warning  ] Can't import `SimaProBlockCSVImporter` - please install `bw2io` with `pip install bw2io[multifunctional]` or install `multifunctional` and `bw_simapro_csv` manually.


In [2]:
config = CONFIG

In [3]:
bd.projects.set_current(config.PROJECT_NAME)

In [4]:
# bi.remote.install_project(project_key="ecoinvent-3.10-biosphere", overwrite_existing=True)

In [5]:
# import import_db

In [6]:
# import_db.import_bonsai(config.BONSAI_PATH)

In [7]:
bonsai = bd.Database('BONSAI_V2.1.6')
bonsai_bio = bd.Database('BONSAI_V2.1.6 biosphere')
ecoinvent_bio = bd.Database('ecoinvent-3.10-biosphere')

### 2. Install foreground REED database

In [8]:
from bw2io.importers import ExcelImporter
from reed.utils import Config

importer  = ExcelImporter(config.REED_PATH/'Model'/'REED_model.xlsx',)
importer.apply_strategies()
importer.match_database(fields=['name','unit','location'])
importer.match_database(db_name=bonsai.name,fields=['name','unit','location','activity type'])
importer.match_database(db_name=bd.config.biosphere,fields=['name','categories','unit'])

Extracted 6 worksheets in 1.29 seconds
Applying strategy: csv_restore_tuples
Applying strategy: csv_restore_booleans
Applying strategy: csv_numerize
Applying strategy: csv_drop_unknown
Applying strategy: csv_add_missing_exchanges_section
Applying strategy: normalize_units
Applying strategy: strip_biosphere_exc_locations
Applying strategy: set_code_by_activity_hash
Applying strategy: link_iterable_by_fields
Applying strategy: assign_only_product_as_production
Applying strategy: link_technosphere_by_activity_hash
Applying strategy: drop_falsey_uncertainty_fields_but_keep_zeros
Applying strategy: convert_uncertainty_types_to_integers
Applying strategy: convert_activity_parameters_to_list
Applied 14 strategies in 0.68 seconds
Applying strategy: link_iterable_by_fields
Applying strategy: link_iterable_by_fields
Applying strategy: link_iterable_by_fields


In [9]:
importer.statistics()

Graph statistics for `reed` importer:
82 graph nodes:
	None: 82
482 graph edges:
	technosphere: 371
	production: 82
	biosphere: 29
482 edges to the following databases:
	BONSAI_V2.1.6: 256
	reed: 197
	ecoinvent-3.10-biosphere: 29
0 unique unlinked edges (0 total):




(82, 482, 0, 0)

In [10]:
# Find all unlinked exchanges
unlinked = [(act, exc) 
            for act in importer.data 
            for exc in act.get('exchanges', [])
            if exc.get('input') is None and exc.get('type') == 'technosphere']

for act, exc in unlinked:
    print(f"Activity: {act.get('name')} | {act.get('location')}")
    print(f"  Unlinked exchange: {exc.get('name')} | {exc.get('unit')} | {exc.get('location')}")
    print()

In [11]:
importer.write_database()
reed = bd.Database('reed')

100%|██████████| 82/82 [00:00<00:00, 372.29it/s]


09:26:10+0200 [info     ] Vacuuming database            
Created database: reed


In [12]:
data_reed = [activity.as_dict() for activity in reed]
df_reed = pd.DataFrame(data_reed)


In [13]:
from IPython.display import display
display(df_reed[['name', 'comment', 'location', 'unit', 'database']])

,name,comment,location,unit,database
0,"Soraluce machining, Komatsu piece",None,GB,unit,reed
1,generic machining,to be used as proxy when machining process is ...,DE,hour,reed
2,boring,None,DE,hour,reed
3,"welding, manual","draft for manual welding process (MIG), missin...",GB,hour,reed
4,Casting,Continious steel casting. Data from a polish f...,DE,ton,reed
...,...,...,...,...,...
77,"Soraluce, average",This inventory is calculated based on informat...,DE,unit,reed
78,WWR total,None,DE,unit,reed
79,"generic machining, soraluce",to be used as proxy when machining process is ...,GB,kilogram,reed
80,GOIMEK A2: transport,None,ES,unit,reed


In [14]:
bonsai = bd.Database('BONSAI_V2.1.6')
bonsai_bio = bd.Database('BONSAI_V2.1.6 biosphere')
ecoinvent = bd.Database('ecoinvent-3.10-biosphere')
reed = bd.Database('reed')

In [15]:
bd.databases

Databases dictionary with 4 object(s):
	BONSAI_V2.1.6
	BONSAI_V2.1.6 biosphere
	ecoinvent-3.10-biosphere
	reed